# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ihSomik/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

### Selected Lane:
**Refresh / Content Opportunity Scoring**

### Task Type:
**Scoring (and Ranking)**

### Why this task type was selected:
Our primary business objective is to look at all of our existing published content and produce a prioritized review queue that helps human editors quickly identify which articles are in critical need of updates or represent high-impact refresh opportunities.

* **Why Scoring?** We want to assign a continuous or probabilistic rating (e.g., `opportunity_score` from `0.0` to `1.0` or `0` to `100`) to every single content asset. This represents the expected benefit (gain in impressions/clicks) or urgency (decay rate of traffic) of updating that specific piece of content.
* **Why Ranking?** The ultimate delivery artifact is a sorted, prioritized queue. While classification would partition pages into discrete buckets (e.g., 'Refresh' vs 'Do Not Refresh'), editors have constrained weekly budgets. Scoring allows us to sort every item globally or per-client, delivering an ordered list where the highest scores are acted on first. This avoids arbitrary thresholds and maximizes the ROI of editorial hours.

In [30]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

data_path = 'flyrank-ml-internship/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
print("Columns related to freshness and traffic decay:")
print(df[['content_age_days', 'days_since_last_update', 'freshness_tier', 'trend_direction', 'trend_pct']].head(5))

Dataset shape: (30000, 44)
Columns related to freshness and traffic decay:
   content_age_days  days_since_last_update freshness_tier trend_direction  \
0               187                      20           0-30            down   
1               445                      25           0-30            down   
2               141                      20           0-30            down   
3               463                      22           0-30          stable   
4               263                      14           0-30            down   

   trend_pct  
0      -41.4  
1      -57.7  
2      -60.9  
3      -13.8  
4      -34.7  


## 2. Target or proxy

### What we would predict:
We predict a continuous value representing **expected search engagement loss (or potential recovery gain)** if the page is left un-refreshed, versus if it is updated.

### Concrete Target Proxy formulation:
We construct our target variable as a continuous numeric proxy based on measured search performance decay:
$$\text{Decay Metric} = \frac{\text{clicks\_last\_30d} - \text{clicks\_prev\_30d}}{\text{clicks\_prev\_30d} + 1}$$
This capture of momentum (i.e. negative trend_pct representing traffic erosion) allows us to identify underperforming or slipping content assets.

### Where does the label come from:
This target is an **observed outcome** derived entirely from our search signals over distinct, non-overlapping historical time windows (e.g., comparing the current 30-day window to the preceding 30-day window). We are **not** predicting subjective manual editorial tags; we use real performance facts which ensures the model learns from actual user interaction trends.

In [31]:
df['target_decay'] = (df['clicks_last_30d'] - df['clicks_prev_30d']) / (df['clicks_prev_30d'] + 1)

print("Summary Statistics of our Target Decay Proxy:")
print(df['target_decay'].describe())

print("\nAverage target decay value by trend direction:")
print(df.groupby('trend_direction')['target_decay'].mean())

Summary Statistics of our Target Decay Proxy:
count    30000.000000
mean         0.147850
std          1.019933
min         -0.987342
25%          0.000000
50%          0.000000
75%          0.000000
max         65.000000
Name: target_decay, dtype: float64

Average target decay value by trend direction:
trend_direction
down      0.053109
flat      0.000000
new       0.337209
stable    0.199846
up        0.370639
Name: target_decay, dtype: float64


## 3. Success metric

### Primary ML Success Metric:
**NDCG@K (Normalized Discounted Cumulative Gain) at K = 100**

### Why we can defend this metric:
Because our end output is a ranked review queue for content editors, the quality of our ordering at the very top of the list is critical. If an editor can only review 50 pages a week, we want the most valuable, highest-decay opportunities to occupy the first 50 slots. NDCG treats ranking as a position-weighted utility:
* A highly decaying, high-volume page placed at rank #1 contributes vastly more to our success than if it is misplaced at rank #90.
* Position-based discounting penalizes errors at the top of the queue much more harshly than errors further down.

### What number means "good":
* **NDCG@100 > 0.85** against the historical, actual measured engagement recovery of refreshed pages.
* Compared to our baseline heuristic (e.g., a simple chronological rule: "sort by days since last update"), our ML model must demonstrate at least a **+15% relative improvement** in NDCG.

In [32]:
baseline_top_10 = df.sort_values(by='days_since_last_update', ascending=False).head(10)[['content_id', 'days_since_last_update', 'target_decay']]
optimal_top_10 = df.sort_values(by='target_decay', ascending=True).head(10)[['content_id', 'days_since_last_update', 'target_decay']]

print("=== Baseline sorting (by days since last update) top 10 target decays ===")
print(baseline_top_10.to_string())

print("\n=== Optimal sorting (by actual target decay) top 10 target decays ===")
print(optimal_top_10.to_string())

=== Baseline sorting (by days since last update) top 10 target decays ===
                 content_id  days_since_last_update  target_decay
26242  content_55a5b1c46474                     373           0.0
4606   content_3f3576c295f5                     373           0.0
29384  content_f6fdf87348f6                     373           0.0
24216  content_1b4ec72dafd4                     372           0.0
18440  content_8d56efff1e71                     372           0.0
6962   content_f01216059a6a                     335           1.0
15608  content_06e19c6486b0                     334           0.0
8631   content_e2b702f4f92b                     334           0.0
21984  content_02b0d6e30129                     313           0.0
15790  content_6476d1d8c050                     313           0.0

=== Optimal sorting (by actual target decay) top 10 target decays ===
                 content_id  days_since_last_update  target_decay
20582  content_0efc49400f2c                     104     -0.9873

## 4. The unit of analysis, as a real dataframe

### What is our unit of analysis?
Our unit of analysis (one row) is a **single unique pseudonymized content asset (`content_id`) belonging to a client (`client_id`) at a specific point in time**.

### Dataframe representation:
Each row contains static metadata of the content (word count, intent, age tier) combined with historical search and session indicators compiled over the past 90 days.

In [33]:
unit_df = df[['content_id', 'client_id', 'search_volume', 'word_count', 'impressions_90d', 'clicks_90d', 'days_since_last_update', 'target_decay']].head(5)
print("Unit of analysis (one row = one content asset for a client):")
display(unit_df)

Unit of analysis (one row = one content asset for a client):


,content_id,client_id,search_volume,word_count,impressions_90d,clicks_90d,days_since_last_update,target_decay
0,content_304f48230142,client_f369cb89fc,10.0,3221.0,3803,29,20,-0.785714
1,content_a1fb4e703a9e,client_4e07408562,90.0,2481.0,15320,7,25,0.500000
2,content_9aa793d4d895,client_7f2253d7e2,0.0,3515.0,12581,11,20,-0.500000
3,content_331d6c4de07b,client_19581e27de,10.0,NaN,11751,58,22,0.277778
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,2803.0,19140,24,14,2.666667


## 5. Why ML beats a fixed rule here

### Why a fixed heuristic rule is insufficient:
An intuitive editorial rule would be: *"If an article has not been updated in 180 days AND traffic is down, put it in the queue."*

However, a simple fixed rule fails in several key scenarios:
1. **Search Seasonality vs. Decay:** A 30% drop in clicks might indicate that an article needs an update, or it could simply be a seasonal dip for a search keyword (e.g., tax-related content outside of April). An ML model can learn to weight the decline relative to the performance of similar content types and search trends.
2. **Varying ROI:** Some articles are high-impact, high-volume evergreen assets, while others target hyper-specific low-volume terms. Re-writing a low-volume page that fell from 5 clicks to 0 clicks has almost zero ROI compared to updating a highly competitive high-intent transactional article that slipped from rank 2 to rank 4. An ML scoring model easily fuses multiple features (CPC, search volume, current ranking positions, decay momentum) to correctly weigh expected ROI.
3. **Non-linear feature interactions:** The relationship between word count, search intent, age tier, impression tier, and traffic erosion is multi-dimensional. Hand-crafting nested `if-else` loops to weight these correctly across different clients is functionally impossible to scale and maintain.

In [34]:
corr_cols = ['search_volume', 'word_count', 'impressions_90d', 'clicks_90d', 'days_since_last_update', 'target_decay']
correlations = df[corr_cols].corr()['target_decay']
print("Linear correlation of signals with target decay:")
print(correlations)

Linear correlation of signals with target decay:
search_volume             0.006406
word_count                0.066549
impressions_90d           0.007248
clicks_90d               -0.010930
days_since_last_update    0.012268
target_decay              1.000000
Name: target_decay, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.